In [ ]:
import os
import sys

sys.path.append(os.path.abspath(os.path.dirname(os.getcwd())))

import math
import json
import itertools
import pandas as pd
import dataclasses
from pathlib import Path
from dataclasses import dataclass
from dataclasses_json import DataClassJsonMixin
from conversion.types import ModelType, ModelPartId, RuntimeParams, ProfileResults, BenchmarkResults
from conversion import full_model_factory, split_full_model, exporter_factory, load_split_model, ModelTester

## Helpers

In [ ]:
@dataclass
class Params:
    name: str
    model_params: dict = dataclasses.field(default_factory=dict)
    split_model_params: dict = dataclasses.field(default_factory=dict)
    exporter_params: dict = dataclasses.field(default_factory=dict)
    model_type: ModelType = ModelType.COREML
    runtime_params: RuntimeParams = RuntimeParams()
    model_path: str = ""


@dataclass
class RunResults:
    run_id: int
    params: Params
    profile: ProfileResults
    benchmark: BenchmarkResults


@dataclass
class SweepResults(DataClassJsonMixin):
    name: str
    results: list[RunResults]

## Parameters

In [ ]:
use_cached_results = False
enable_profiling = False
num_repeats = 3

# -------------------------------------------------------------------------------------
# Model benchmark study
# -------------------------------------------------------------------------------------

benchmark_name = "model_profile"
use_decoder = True
benchmark_params = [
    # Params("mlvc1a", dict(model_version="mlvc1a")),
    # DMC 6.1
    # Params("dmc61", dict(model_version="dmc61")),
    # Params("dmc61_silu", dict(model_version="dmc61_silu")),
    # Params("dmc61_relu", dict(model_version="dmc61_silu", activation="ReLU")),
    # Params("dmc61_lrelu", dict(model_version="dmc61_silu", activation="LeakyReLU")),
    # DMC 6.1r
    # Params("dmc61r_silu", dict(model_version="dmc61r_silu")),
    # Params("dmc61r_lrelu", dict(model_version="dmc61r_silu", activation="LeakyReLU")),
    # DMC 6.1s
    # Params("dmc61s", dict(model_version="dmc61s")),
    # Params("dmc61s_silu", dict(model_version="dmc61s_silu")),
    # Params("dmc61s_lrelu", dict(model_version="dmc61s_lrelu")),
    # DMC 6.1sb
    # Params("dmc61sb", dict(model_version="dmc61sb")),
    Params("dmc61sb_silu", dict(model_version="dmc61sb_silu")),
    Params("dmc61sb_lrelu", dict(model_version="dmc61sb_lrelu")),
    Params("dmc61sbr_lrelu", dict(model_version="dmc61sbr_lrelu")),
    Params("dmc61sbr_mini_reglu", dict(model_version="dmc61sbr_mini_reglu")),
]

# -------------------------------------------------------------------------------------

results_file = Path(f"./output/benchmark/results/{benchmark_name}_{num_repeats}_{enable_profiling:d}.json")

## Prepare converted models

In [ ]:
for params in benchmark_params:
    full_model = full_model_factory(**params.model_params)  # type: ignore
    split_model = split_full_model(full_model, model_width=960, model_height=544, **params.split_model_params)  # type: ignore
    exporter = exporter_factory(
        split_model=split_model,
        model_type=params.model_type,
        output_path="./output/benchmark/models",
        output_name=params.name,
        skip_if_exists=use_cached_results,
        frame_count=1,
        q_index_list=[63],
        **params.exporter_params,
    )
    params.model_path = str(exporter.run())

## Benchmark models

In [ ]:
if use_cached_results and results_file.exists():
    with open(results_file, "r") as f:
        sweep_results = SweepResults.from_dict(json.load(f))
    print(f"Loaded cached results from {results_file}")
else:
    results: list[RunResults] = []
    for run_id, params in itertools.product(range(num_repeats), benchmark_params):
        print(f"Running benchmark for {params.name} (run {run_id + 1}/{num_repeats})")
        split_model = load_split_model(params.model_path, use_decoder=use_decoder, runtime_params=params.runtime_params)
        tester = ModelTester(split_model)
        if enable_profiling:
            profile = tester.profile()
        else:
            profile = ProfileResults(profiles={})
        benchmark = tester.benchmark(use_decoder=use_decoder, q_index=63, frame_count=300)

        results.append(
            RunResults(
                params=params,
                run_id=run_id,
                profile=profile,
                benchmark=benchmark,
            )
        )
    sweep_results = SweepResults(
        name=benchmark_name,
        results=results,
    )
    results_file.parent.mkdir(parents=True, exist_ok=True)
    with open(results_file, "w") as f:
        json.dump(sweep_results.to_dict(), f, indent=4)

## Visualize results

In [ ]:
model_part_short_names = {
    "MLVCEncoder": "E1",
    "MLVCEncoderPart1": "E1",
    "MLVCEncoderPart2": "E2",
    "MLVCDecoder": "D1",
    "MLVCDecoderPart1": "D1",
    "MLVCDecoderPart2": "D2",
    "MLVCDecoderPart3": "D3",
}

with open(results_file, "r") as f:
    sweep_results = SweepResults.from_dict(json.load(f))
print(f"Name: {sweep_results.name}")

df_res = []
for run_data in sweep_results.results:
    params = run_data.params
    benchmark = run_data.benchmark
    profile = run_data.profile
    timers = benchmark.frame_loop_summary.timers

    if "MLVCEncoderPart1" in timers:
        enc_latency1 = timers["MLVCEncoderPart1"].median
        enc_latency2 = timers["MLVCEncoderPart2"].median
        enc_fps = 1.0 / (enc_latency1 + enc_latency2 / 64.0)
    else:
        enc_fps = 1.0 / timers["MLVCEncoder"].median

    if "MLVCDecoderTotal" in timers:
        dec_fps = 1.0 / timers["MLVCDecoderTotal"].median
    else:
        dec_fps = float("nan")

    if not math.isnan(dec_fps):
        total_fps = 1.0 / (1.0 / enc_fps + 1.0 / dec_fps)
    else:
        total_fps = enc_fps

    r = {
        ("", "model"): params.name,
        ("", "run_id"): run_data.run_id,
        ("fps", "enc"): round(enc_fps, 1),
        ("fps", "dec"): round(dec_fps, 1),
        ("fps", "total"): round(total_fps, 1),
        ("compression", "psnr"): round(benchmark.frame_loop_summary.psnr.mean, 3),
        ("compression", "bpp"): round(benchmark.frame_loop_summary.bpp.mean, 5),
    }

    # Model part latencies
    for timer_name, short_name in model_part_short_names.items():
        if timer_name not in timers:
            continue
        timer_value = timers[timer_name]
        r.update({("latency_ms", short_name): round(1e3 * timer_value.median, 1)})

    # Profile results
    if ModelPartId.ENCODER_PART1 in profile.profiles:
        profile1 = profile.profiles[ModelPartId.ENCODER_PART1]
        profile2 = profile.profiles[ModelPartId.ENCODER_PART2]
    elif ModelPartId.ENCODER in profile.profiles:
        profile1 = profile.profiles[ModelPartId.ENCODER]
        profile2 = None
    else:
        profile1 = None
        profile2 = None

    if profile1 is not None:
        r.update(
            {
                ("cpu_ops", "enc1"): profile1.op_stats.num_ops_cpu,
                ("cpu_ops", "enc2"): profile2.op_stats.num_ops_cpu if profile2 else 0,
                ("npu_ops", "enc1"): profile1.op_stats.num_ops_npu,
                ("npu_ops", "enc2"): profile2.op_stats.num_ops_npu if profile2 else 0,
            }
        )

    # Idle power
    idle_pwr = benchmark.powermetrics_data.stats.get("wait_before_loop")
    if idle_pwr is not None:
        r.update(
            {
                ("idle_power", "cpu"): round(idle_pwr.cpu_power.median, 1),
                ("idle_power", "gpu"): round(idle_pwr.gpu_power.median, 1),
                ("idle_power", "npu"): round(idle_pwr.npu_power.median, 1),
            }
        )

    # Frame loop power
    pwr = benchmark.powermetrics_data.stats.get("frame_loop")
    if pwr is not None:
        r.update(
            {
                ("power", "cpu"): round(pwr.cpu_power.median, 1),
                ("power", "gpu"): round(pwr.gpu_power.median, 1),
                ("power", "npu"): round(pwr.npu_power.median, 1),
                ("power", "N"): pwr.cpu_power.count,
            }
        )

    df_res.append(r)

df_res = pd.DataFrame.from_records(df_res)
df_res.columns = pd.MultiIndex.from_tuples(df_res.columns)
df_res.sort_values([("", "model"), ("", "run_id")])  # type: ignore

In [ ]:
best_runs = df_res.groupby([("", "model")])[[("fps", "total")]].idxmax().squeeze()  # type: ignore
# best_runs = df_res.groupby([("", "model")])[[("fps", "enc")]].apply(lambda x: (x - x.median()).abs().idxmin()).squeeze()  # type: ignore
df_best = df_res.loc[best_runs]  # type: ignore
df_best

In [ ]:
print(df_best.to_csv(index=False, sep="\t"))